In [ ]:
# Green Fleet Optimizer: SIH 2026 Demo

This notebook runs the core end-to-end demonstration on synthetic maritime data: fuel prediction with uncertainty, quantum-inspired versus classical optimization, compliance-aware plan inspection, and operational intelligence.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
BACKEND = PROJECT_ROOT / "backend"
sys.path.insert(0, str(BACKEND))

from data.synthetic_generator import run as generate_synthetic_data

RAW = BACKEND / "data" / "raw"
PROCESSED = BACKEND / "data" / "processed"
if not (RAW / "vessels.json").exists() or not (PROCESSED / "voyage_dataset.csv").exists():
    generate_synthetic_data()

voyages = pd.read_csv(PROCESSED / "voyage_dataset.csv")
print(f"Synthetic voyages: {len(voyages):,}")
voyages.head()

In [ ]:
from prediction.predictor import (
    predict_fuel_consumption_with_uncertainty,
    calculate_wtw_emissions,
    calculate_cii_rating,
)

sample = voyages.iloc[0]
prediction = predict_fuel_consumption_with_uncertainty(
    vessel_type=sample.vessel_type,
    capacity=sample.capacity,
    engine_power_kw=sample.engine_power_kw,
    distance_nmi=sample.distance_nmi,
    speed_knots=sample.speed_knots,
    sea_state=sample.sea_state,
    payload_pct=sample.payload_pct,
    fuel_type=sample.fuel_type,
)
print("Prediction with 90% confidence band:")
print(prediction)
print("CII:", calculate_cii_rating(
    calculate_wtw_emissions(prediction["fuel_consumption_tonnes"], sample.fuel_type),
    sample.capacity,
    sample.distance_nmi,
))

In [ ]:
from optimization.optimizer import run_fleet_optimization

with open(RAW / "vessels.json") as handle:
    vessels = json.load(handle)
with open(RAW / "routes.json") as handle:
    routes = json.load(handle)

optimization = run_fleet_optimization(
    vessels=vessels[:8],
    routes=routes,
    algorithm="QIGA",
    pop_size=12,
    generations=5,
    carbon_tax=100,
    demand_mult=0.8,
    allowed_fuels=["HFO", "LNG", "Methanol", "Ammonia"],
    shore_power_enabled=True,
)
print(optimization["primary_algorithm"], optimization["total_execution_time_seconds"], "seconds")
pareto = pd.DataFrame(optimization["result"]["pareto_front"])
pareto[["cost_usd", "emissions_tco2e", "delay_hours"]].head()

In [ ]:
from benchmarking.metrics import summarize_algorithm_performance

comparison = run_fleet_optimization(
    vessels=vessels[:6],
    routes=routes,
    algorithm="ALL",
    pop_size=8,
    generations=3,
    carbon_tax=100,
)
benchmark_rows = [summarize_algorithm_performance(result) for result in comparison["all_results"].values()]
benchmark = pd.DataFrame(benchmark_rows)
benchmark[["algorithm", "hypervolume_score", "convergence_generation", "execution_time_seconds", "pareto_solutions_count"]]

In [ ]:
plt.figure(figsize=(8, 4))
for algorithm, group in benchmark.groupby("algorithm"):
    result = comparison["all_results"][algorithm.split()[0]] if algorithm.split()[0] in comparison["all_results"] else None
    if result:
        history = pd.DataFrame(result["convergence_history"])
        plt.plot(history.index + 1, history["min_cost"], marker="o", label=algorithm)
plt.title("Convergence: shared fleet and routes")
plt.xlabel("Generation")
plt.ylabel("Best cost (USD)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
from api.routes.scenario import (
    WeatherUpdate,
    weather_reroute,
    bunkering_recommendations,
    retrofit_roi,
)

weather = weather_reroute(WeatherUpdate(route_id="R03", new_sea_state=5.5))
print("Weather-adaptive route:", weather["recommended_route"]["id"])
print("Speed:", weather["recommended_speed_knots"], "knots")
print("Fuel delta:", weather["fuel_delta_tonnes"], "tonnes")

bunkering = bunkering_recommendations(route_id="R03", fuel_type="LNG")
print("Recommended bunker:", bunkering["recommendation"]["port"])
print("Arbitrage:", bunkering["arbitrage_saving_usd_t"], "USD/t")

roi = retrofit_roi()
pd.DataFrame(roi["ranked_options"])